# 12 — Comparar filtros por metadatos en la consulta final

Medimos si filtrar por `modulo` o por `tipo_pagina` mejora la recuperación de `ConsultorRAG` (colección `pydocs_nomic_est1500_m200_sinruta`, 20 candidatos, reranker `cross-encoder/ms-marco-MiniLM-L-6-v2`, k=4) sobre las 32 preguntas A-D. Son nueve configuraciones: sin filtro y, por cada campo, filtro perfecto y filtro equivocado, cada uno sin repetición (`umbral=None`) y con repetición sin filtro (`umbral=0.0`).

## Criterio de decisión (RF-10), fijado antes de medir

- Se recomienda al agente pasar el filtro de un campo si, con filtro perfecto y sin repetición, ev_hit@4 llega al menos a 27/32 y hit@4 por fuente no baja de 30/32.
- La repetición sin filtro con umbral 0,0 se mantiene si, con filtro equivocado y repetición, ev_hit@4 no baja de 23/32 en ese campo. Si baja, se registra como riesgo; el umbral no se ajusta con el golden set en este cambio.
- Si el filtro perfecto con repetición sube la latencia total media más de un 10 % frente a sin filtro, se dice junto a la recomendación.

Comprobación previa (RNF-04): la configuración «sin filtro» debe reproducir `validacion_final.csv` (30/32 fuente y 26/32 evidencia en el top 4). Si no, el notebook para.

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    calcular_metricas, leer_evidencias, leer_golden_set,
    rango_primer_acierto, rango_primer_acierto_evidencia,
)
from consultar_rag import COLECCION, ConsultorRAG
from metadatos import modulo_de_fichero, tipo_pagina

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
VALIDACION = RAIZ / "data" / "resultados" / "validacion_final.csv"
SALIDA_RESUMEN = RAIZ / "data" / "resultados" / "filtros_metadatos.csv"
SALIDA_PREGUNTA = RAIZ / "data" / "resultados" / "filtros_metadatos_por_pregunta.csv"
CANDIDATOS = 20
K = 4
UMBRAL = 0.0

# Criterio de RF-10 (el mismo que la celda de markdown inicial)
EV_HIT_PERFECTO_MIN = 27
HIT_PERFECTO_MIN = 30
EV_HIT_EQUIVOCADO_MIN = 23
LATENCIA_MAX_SUBIDA = 0.10

print("Colección:", COLECCION)

Colección: pydocs_nomic_est1500_m200_sinruta


## 3. Conexión

`ConsultorRAG.local()` comprueba Qdrant, los metadatos de la colección, Ollama y el reranker, y para con un mensaje si algo falta.

In [2]:
consultor = ConsultorRAG.local()
print("Consultor listo")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Consultor listo


## 4. Golden set y evidencias

In [3]:
preguntas = [p for p in leer_golden_set(GOLDEN) if p["categoria"] != "E"]
evidencias = leer_evidencias(EVIDENCIAS)
print("Preguntas A-D:", len(preguntas))

Preguntas A-D: 32


## 5. Filtros perfectos y equivocados de cada pregunta

Reglas de RF-09. El perfecto por `modulo` son los `modulo` no vacíos de las fuentes esperadas; el perfecto por `tipo_pagina`, los `tipo_pagina` no vacíos. El equivocado por `modulo` es el perfecto de la siguiente pregunta, en orden circular, con un perfecto distinto; el equivocado por `tipo_pagina` es `c-api` si el perfecto incluye `library` y `library` en el resto. Sin filtro perfecto, la pregunta va sin filtro en esa configuración. La tabla se muestra antes de medir.

In [4]:
def sin_repetidos(valores):
    return list(dict.fromkeys(v for v in valores if v))

perfecto_modulo = [sin_repetidos(modulo_de_fichero(CARPETA_CORPUS, f) for f in p["fuentes"]) for p in preguntas]
perfecto_tipo = [sin_repetidos(tipo_pagina(f) for f in p["fuentes"]) for p in preguntas]

def equivocado_modulo(i):
    if not perfecto_modulo[i]:
        return []
    n = len(preguntas)
    for paso in range(1, n):
        otro = perfecto_modulo[(i + paso) % n]
        if otro and set(otro) != set(perfecto_modulo[i]):
            return otro
    return []

def equivocado_tipo(i):
    if not perfecto_tipo[i]:
        return []
    return ["c-api"] if "library" in perfecto_tipo[i] else ["library"]

equivocado_modulo_l = [equivocado_modulo(i) for i in range(len(preguntas))]
equivocado_tipo_l = [equivocado_tipo(i) for i in range(len(preguntas))]

tabla_filtros = pd.DataFrame({
    "id": [p["id"] for p in preguntas],
    "perfecto modulo": [", ".join(v) for v in perfecto_modulo],
    "equivocado modulo": [", ".join(v) for v in equivocado_modulo_l],
    "perfecto tipo_pagina": [", ".join(v) for v in perfecto_tipo],
    "equivocado tipo_pagina": [", ".join(v) for v in equivocado_tipo_l],
})
print("Preguntas con filtro perfecto por modulo:", sum(bool(v) for v in perfecto_modulo), "de", len(preguntas))
tabla_filtros

Preguntas con filtro perfecto por modulo: 22 de 32


,id,perfecto modulo,equivocado modulo,perfecto tipo_pagina,equivocado tipo_pagina
0,A01,http.client,urllib.request,library,c-api
1,A02,urllib.request,pathlib,library,c-api
2,A03,pathlib,re,library,c-api
3,A04,re,json,library,c-api
4,A05,,,builtins,library
5,A06,json,subprocess,library,c-api
6,A07,subprocess,sqlite3,library,c-api
7,A08,sqlite3,argparse,library,c-api
8,A09,,,tutorial,library
9,A10,argparse,http,library,c-api


## 6. Configuraciones

In [5]:
# (nombre, campo, lista de valores por pregunta o None, umbral)
CONFIGURACIONES = [
    ("1 sin filtro", "", None, UMBRAL),
    ("2 perfecto modulo sin repeticion", "modulo", perfecto_modulo, None),
    ("3 perfecto modulo con repeticion", "modulo", perfecto_modulo, UMBRAL),
    ("4 equivocado modulo sin repeticion", "modulo", equivocado_modulo_l, None),
    ("5 equivocado modulo con repeticion", "modulo", equivocado_modulo_l, UMBRAL),
    ("6 perfecto tipo_pagina sin repeticion", "tipo_pagina", perfecto_tipo, None),
    ("7 perfecto tipo_pagina con repeticion", "tipo_pagina", perfecto_tipo, UMBRAL),
    ("8 equivocado tipo_pagina sin repeticion", "tipo_pagina", equivocado_tipo_l, None),
    ("9 equivocado tipo_pagina con repeticion", "tipo_pagina", equivocado_tipo_l, UMBRAL),
]
for c in CONFIGURACIONES:
    print(c[0])

1 sin filtro
2 perfecto modulo sin repeticion
3 perfecto modulo con repeticion
4 equivocado modulo sin repeticion
5 equivocado modulo con repeticion
6 perfecto tipo_pagina sin repeticion
7 perfecto tipo_pagina con repeticion
8 equivocado tipo_pagina sin repeticion
9 equivocado tipo_pagina con repeticion


## 7. Medición

Una fila por configuración y pregunta. Una consulta previa calienta el reranker y no se mide.

In [6]:
consultor.consultar(preguntas[0]["query"])  # calentamiento

filas = []
for nombre, campo, valores, umbral in tqdm(CONFIGURACIONES, desc="configuraciones"):
    for i, pregunta in enumerate(preguntas):
        valor = valores[i] if valores is not None else []
        filtros = {campo: valor} if valor else None
        inicio = time.perf_counter()
        resultados = consultor.consultar(pregunta["query"], candidatos=CANDIDATOS, k=K, filtros=filtros, umbral=umbral)
        latencia_ms = (time.perf_counter() - inicio) * 1000
        rango = rango_primer_acierto([r["fuente"] for r in resultados], pregunta["fuentes"])
        rango_ev = rango_primer_acierto_evidencia(resultados, evidencias[pregunta["id"]])
        filas.append({
            "configuracion": nombre,
            "campo": campo,
            "filtro": "|".join(valor) if filtros else "",
            "repetida_sin_filtro": bool(filtros) and not any(r["filtrado"] for r in resultados),
            "id": pregunta["id"],
            "categoria": pregunta["categoria"],
            "rango fuente": rango,
            "rango evidencia": rango_ev,
            "hit@4": rango is not None and rango <= K,
            "ev_hit@4": rango_ev is not None and rango_ev <= K,
            "latencia total (ms)": latencia_ms,
        })
por_pregunta = pd.DataFrame(filas)
len(por_pregunta)

configuraciones:   0%|          | 0/9 [00:00<?, ?it/s]

288

## 8. Comprobación de «sin filtro» contra `validacion_final.csv` (RNF-04)

Si alguna pregunta no coincide, se imprime pregunta a pregunta y la celda para sin guardar nada.

In [7]:
validacion = pd.read_csv(VALIDACION).set_index("id")
base = por_pregunta[por_pregunta["configuracion"] == "1 sin filtro"].set_index("id")
diferencias = []
for id_, fila in base.iterrows():
    for columna in ("rango fuente", "rango evidencia"):
        esperado = validacion.loc[id_, columna]
        obtenido = fila[columna]
        if not (pd.isna(esperado) and pd.isna(obtenido)) and esperado != obtenido:
            diferencias.append((id_, columna, esperado, obtenido))
for d in diferencias:
    print("DIFERENCIA", *d)
n_fuente = int(base["hit@4"].sum())
n_ev = int(base["ev_hit@4"].sum())
print(f"sin filtro: {n_fuente}/32 fuente, {n_ev}/32 evidencia; diferencias con validacion_final.csv: {len(diferencias)}")
assert not diferencias and (n_fuente, n_ev) == (30, 26), "RNF-04 no se cumple: no se guarda nada"

sin filtro: 30/32 fuente, 26/32 evidencia; diferencias con validacion_final.csv: 0


## 9. Métricas por configuración

In [8]:
def metrica(df, columna):
    rangos = [None if pd.isna(v) else int(v) for v in df[columna]]
    return calcular_metricas(rangos, K)

resumen_filas = []
for nombre, _, _, _ in CONFIGURACIONES:
    df = por_pregunta[por_pregunta["configuracion"] == nombre]
    fuente = metrica(df, "rango fuente")
    evidencia = metrica(df, "rango evidencia")
    resumen_filas.append({
        "configuracion": nombre,
        "preguntas filtradas": int((df["filtro"] != "").sum()),
        "repeticiones": int(df["repetida_sin_filtro"].sum()),
        "hit@1": fuente["hit@1"],
        "hit@4": fuente["hit@4"],
        "MRR": fuente["mrr"],
        "ev_hit@1": evidencia["hit@1"],
        "ev_hit@4": evidencia["hit@4"],
        "ev_MRR": evidencia["mrr"],
        "latencia media (ms)": df["latencia total (ms)"].mean(),
    })
resumen = pd.DataFrame(resumen_filas)
resumen

,configuracion,preguntas filtradas,repeticiones,hit@1,hit@4,MRR,ev_hit@1,ev_hit@4,ev_MRR,latencia media (ms)
0,1 sin filtro,0,0,0.68750,0.93750,0.796875,0.43750,0.81250,0.604167,1018.804834
1,2 perfecto modulo sin repeticion,22,0,0.87500,1.00000,0.937500,0.56250,0.90625,0.710938,892.774212
2,3 perfecto modulo con repeticion,22,1,0.84375,0.96875,0.906250,0.56250,0.87500,0.700521,900.502538
3,4 equivocado modulo sin repeticion,22,0,0.18750,0.31250,0.250000,0.09375,0.31250,0.197917,873.966547
4,5 equivocado modulo con repeticion,22,20,0.65625,0.90625,0.765625,0.43750,0.81250,0.604167,1459.320641
5,6 perfecto tipo_pagina sin repeticion,32,0,0.84375,0.93750,0.880208,0.59375,0.81250,0.687500,1004.150762
6,7 perfecto tipo_pagina con repeticion,32,1,0.84375,0.93750,0.880208,0.59375,0.81250,0.687500,1032.989722
7,8 equivocado tipo_pagina sin repeticion,32,0,0.00000,0.00000,0.000000,0.00000,0.00000,0.000000,957.038963
8,9 equivocado tipo_pagina con repeticion,32,21,0.50000,0.59375,0.531250,0.31250,0.46875,0.375000,1576.779397


## 10. Guardar los CSV

In [9]:
SALIDA_RESUMEN.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(SALIDA_RESUMEN, index=False)
por_pregunta.to_csv(SALIDA_PREGUNTA, index=False)
print("Guardado", SALIDA_RESUMEN.relative_to(RAIZ))
print("Guardado", SALIDA_PREGUNTA.relative_to(RAIZ))

Guardado data\resultados\filtros_metadatos.csv
Guardado data\resultados\filtros_metadatos_por_pregunta.csv


## 11. Criterio de RF-10 aplicado

Se aplica tal cual está escrito arriba; los umbrales vienen de la celda de configuración.

In [10]:
def fila(nombre):
    return resumen[resumen["configuracion"] == nombre].iloc[0]

def aciertos(valor):
    return round(valor * len(preguntas))

base_lat = fila("1 sin filtro")["latencia media (ms)"]
decision = []
for campo, perfecto_sin, perfecto_con, equiv_con in [
    ("modulo", "2 perfecto modulo sin repeticion", "3 perfecto modulo con repeticion", "5 equivocado modulo con repeticion"),
    ("tipo_pagina", "6 perfecto tipo_pagina sin repeticion", "7 perfecto tipo_pagina con repeticion", "9 equivocado tipo_pagina con repeticion"),
]:
    p = fila(perfecto_sin)
    ev, h = aciertos(p["ev_hit@4"]), aciertos(p["hit@4"])
    ev_eq = aciertos(fila(equiv_con)["ev_hit@4"])
    subida = fila(perfecto_con)["latencia media (ms)"] / base_lat - 1
    decision.append({
        "campo": campo,
        f"perfecto sin repeticion: ev_hit@4 (min {EV_HIT_PERFECTO_MIN})": ev,
        f"perfecto sin repeticion: hit@4 (min {HIT_PERFECTO_MIN})": h,
        "recomendar filtro": ev >= EV_HIT_PERFECTO_MIN and h >= HIT_PERFECTO_MIN,
        f"equivocado con repeticion: ev_hit@4 (min {EV_HIT_EQUIVOCADO_MIN})": ev_eq,
        "mantener repeticion": ev_eq >= EV_HIT_EQUIVOCADO_MIN,
        "subida de latencia con repeticion": f"{subida:+.1%}",
        "avisar latencia (>10 %)": subida > LATENCIA_MAX_SUBIDA,
    })
pd.DataFrame(decision)

,campo,perfecto sin repeticion: ev_hit@4 (min 27),perfecto sin repeticion: hit@4 (min 30),recomendar filtro,equivocado con repeticion: ev_hit@4 (min 23),mantener repeticion,subida de latencia con repeticion,avisar latencia (>10 %)
0,modulo,29,32,True,26,True,-11.6%,False
1,tipo_pagina,26,30,False,15,False,+1.4%,False
